# 미국 수출 예측 - 성장률 종합 분석 (Long-Format)

`us_trade_export_monthly_with_forecast` 테이블(HS 코드별 실측 `expDlr` + 예측 `expDlr_forecast`)을 기반으로,
HS 코드별 최신 실측월을 **기준월**로 삼아 아래 7개 성장률 지표를 계산하고, 하나의 **long-format DataFrame**으로 통합합니다.

## 지표 정의

| metric_type | 정의 |
|---|---|
| `past_12m_growth` | 기준월 포함 최근 12개월 합 vs 그 이전 12개월 합 (완전 과거 vs 과거) |
| `future_12m_growth` | 기준월 이후 12개월(예측 포함) 합 vs 기준월 포함 최근 12개월 합 |
| `ytd_growth` (당해 예상성장률) | 올해 1월~기준월까지 누적 합 vs 작년 동기간(1월~같은 월) 합 |
| `full_year_growth` (올해 예상성장률) | 올해 1~12월 전체(실측+예측 채움) 합 vs 작년 1~12월 전체 합 |
| `last_year_growth` | 작년 1~12월 합 vs 재작년 1~12월 합 (완전 확정 과거 성장률) |
| `yoy_growth` | 특정월 값 vs 작년 같은 달 값 (기준월 전후 12개월 구간, 월별 시계열) |
| `mom_growth` | 특정월 값 vs 전월 값 (기준월 전후 12개월 구간, 월별 시계열) |

> 정의가 의도와 다르면 `metric_type`별 계산 구간(윈도우)만 바꾸면 되도록 함수를 구간 단위로 분리해뒀습니다.

## Long-format 스키마

```
hs_code | reference_date | metric_type | metric_date | period_label |
base_value | compare_value | growth_rate | flag
```

- `base_value` : 비교 기준값(분모), `compare_value` : 비교 대상값(분자)
- `growth_rate` : `(compare_value / base_value - 1) * 100`  (단위: %)
- `flag` : 위생 체크 (`base<=0`, `negative_value`, `explosive`, `missing`, `incomplete_year` 등). `None`이면 정상.
- `past_12m_growth` / `last_year_growth` 는 완전 과거 확정치이므로 `flag`가 거의 발생하지 않고,
  `future_12m_growth` / `full_year_growth` 는 예측치가 섞여있어 `explosive` 플래그가 상대적으로 자주 나올 수 있습니다.


In [1]:

import sys
import os
from pathlib import Path


def setup_universal_paths() -> dict:
    """현재 위치에서 상위로 올라가며 DATA 폴더를 찾아 sys.path 에 등록"""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print("=" * 70)
            print("경로 설정 완료")
            print("=" * 70)
            print(f"  프로젝트 루트 : {parent}")
            print(f"  DATA 폴더     : {data_folder}")
            print(f"  현재 위치     : {current}")
            print("=" * 70 + "\n")
            return {"project_root": parent, "data_folder": data_folder, "current": current}
    raise FileNotFoundError(f"DATA 폴더를 찾을 수 없습니다.\n현재 위치: {current}")


try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    sys.exit(1)

# get_db_host() 등 공통 함수 import
from DATA.stock_invest_function import *


경로 설정 완료
  프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
  DATA 폴더     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
  현재 위치     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\미국_수출데이터_예측



In [2]:

import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text


def _detect_version_col(conn, table):
    """테이블에 created_at / input_date 중 어떤 버전 컬럼이 있는지 감지."""
    res = conn.execute(text(f"SHOW COLUMNS FROM {table}"))
    cols = [r[0] for r in res.fetchall()]
    for c in ("created_at", "input_date"):
        if c in cols:
            return c
    return None


def _load_latest_export_data(engine, table="us_trade_export_monthly_with_forecast"):
    """HS별 '최신 버전(run)'만 로드 + 실측/예측 통합 export_value 생성"""
    with engine.connect() as conn:
        vcol = _detect_version_col(conn, table)
        if vcol:
            sql = f"""
                SELECT t.hs_code, t.date_month_end, t.expDlr,
                       t.expDlr_forecast, t.is_forecast
                FROM {table} t
                JOIN (
                    SELECT hs_code, MAX({vcol}) AS mx
                    FROM {table}
                    GROUP BY hs_code
                ) l ON t.hs_code = l.hs_code AND t.{vcol} = l.mx
                ORDER BY t.hs_code, t.date_month_end
            """
            print(f"[버전 관리] '{vcol}' 기준 HS별 최신 run 만 사용")
        else:
            sql = f"""
                SELECT hs_code, date_month_end, expDlr,
                       expDlr_forecast, is_forecast
                FROM {table}
                ORDER BY hs_code, date_month_end
            """
            print("[주의] 버전 컬럼(created_at/input_date) 없음 -> 전체 행 사용")

        res  = conn.execute(text(sql))
        rows = res.fetchall()
        cols = list(res.keys())

    df = pd.DataFrame(rows, columns=cols)
    df["date_month_end"] = pd.to_datetime(df["date_month_end"])
    df["export_value"] = np.where(df["is_forecast"] == 1,
                                   df["expDlr_forecast"], df["expDlr"])
    return df


def _sum_window(df_hs, start, end):
    """df_hs(단일 hs_code) 에서 [start, end] 구간 합계와 개월 수 반환"""
    mask = (df_hs["date_month_end"] >= start) & (df_hs["date_month_end"] <= end)
    sub = df_hs.loc[mask]
    if sub.empty:
        return np.nan, 0
    return sub["export_value"].sum(), len(sub)


def _growth(base, comp, sanity_ratio=1000.0):
    """growth = (comp/base - 1) * 100, 위생 체크(flag) 포함"""
    if pd.isna(base) or pd.isna(comp):
        return np.nan, "missing"
    if base <= 0:
        return np.nan, "base<=0"
    if comp < 0:
        return np.nan, "negative_value"
    g = (comp / base - 1.0) * 100.0
    if comp > base * sanity_ratio:
        return g, "explosive"
    return g, None


def _month_end(ts):
    """pd.DateOffset(months=N) 연산 후 '진짜 월말'로 정규화.
    (예: 2025-06-30 - 1개월 = 2025-05-30 이 되는 DateOffset 특성상의
     어긋남을 막기 위해, 매 연산 직후 이 함수로 스냅 처리한다.)"""
    return pd.Timestamp(ts) + pd.offsets.MonthEnd(0)


def build_export_growth_long(db_info, reference_date=None, window_months=12,
                              save_path=None, sanity_ratio=1000.0,
                              table="us_trade_export_monthly_with_forecast"):
    """
    HS 코드별 성장률 7종을 계산해 하나의 long-format DataFrame으로 반환.

    Parameters
    ----------
    db_info : dict
        {'host','port','user','password','database'}
    reference_date : str or None
        기준월(YYYY-MM-DD). None이면 HS코드별로 '실측 데이터의 마지막 달'을
        자체 기준월로 사용 (HS코드마다 실측 마지막 달이 다를 수 있음).
    window_months : int
        과거/향후 성장률 계산에 사용할 개월 수 (기본 12개월)
    save_path : str or None
        지정 시 long_df 를 CSV 로 저장

    Returns
    -------
    long_df : pd.DataFrame
        columns = [hs_code, reference_date, metric_type, metric_date,
                   period_label, base_value, compare_value, growth_rate, flag]
    """
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info.get('port', 3306)}/{db_info['database']}"
    )

    print("=" * 80)
    print("수출 성장률 종합 분석 (long-format)")
    print("=" * 80)
    print("데이터 로딩 중...")
    df_all = _load_latest_export_data(engine, table)
    engine.dispose()
    print(f"  로드 레코드: {len(df_all):,} / HS 코드: {df_all['hs_code'].nunique():,}")

    rows = []
    hs_codes = df_all["hs_code"].unique()

    for hs in hs_codes:
        df_hs = df_all[df_all["hs_code"] == hs].sort_values("date_month_end")

        # ── 기준월(reference_date) 결정 ──
        if reference_date is not None:
            ref_dt = pd.to_datetime(reference_date)
        else:
            actual_dates = df_hs.loc[df_hs["is_forecast"] == 0, "date_month_end"]
            if actual_dates.empty:
                continue
            ref_dt = actual_dates.max()

        # ── 1. 과거 N개월 성장률: 최근 N개월(recent) vs 그 이전 N개월(prior) ──
        recent_start = _month_end(ref_dt - pd.DateOffset(months=window_months - 1))
        recent_sum, _ = _sum_window(df_hs, recent_start, ref_dt)

        prior_end   = _month_end(recent_start - pd.DateOffset(months=1))
        prior_start = _month_end(prior_end - pd.DateOffset(months=window_months - 1))
        prior_sum, _ = _sum_window(df_hs, prior_start, prior_end)

        g, flag = _growth(prior_sum, recent_sum, sanity_ratio)
        rows.append(dict(
            hs_code=hs, reference_date=ref_dt, metric_type="past_12m_growth",
            metric_date=ref_dt,
            period_label=f"{recent_start:%Y-%m}~{ref_dt:%Y-%m} vs {prior_start:%Y-%m}~{prior_end:%Y-%m}",
            base_value=prior_sum, compare_value=recent_sum,
            growth_rate=g, flag=flag,
        ))

        # ── 2. 향후 N개월 성장률: future N(예측 포함) vs recent N ──
        future_start = _month_end(ref_dt + pd.DateOffset(months=1))
        future_end   = _month_end(future_start + pd.DateOffset(months=window_months - 1))
        future_sum, _ = _sum_window(df_hs, future_start, future_end)

        g, flag = _growth(recent_sum, future_sum, sanity_ratio)
        rows.append(dict(
            hs_code=hs, reference_date=ref_dt, metric_type="future_12m_growth",
            metric_date=future_end,
            period_label=f"{future_start:%Y-%m}~{future_end:%Y-%m} vs {recent_start:%Y-%m}~{ref_dt:%Y-%m}",
            base_value=recent_sum, compare_value=future_sum,
            growth_rate=g, flag=flag,
        ))

        # ── 3. 당해(YTD) 예상성장률: 올해 1월~기준월 vs 작년 1월~같은 월 ──
        cur_year = ref_dt.year
        ytd_start = pd.Timestamp(year=cur_year, month=1, day=1)
        ytd_end   = ref_dt
        ytd_sum, _ = _sum_window(df_hs, ytd_start, ytd_end)

        py_ytd_start = ytd_start - pd.DateOffset(years=1)
        py_ytd_end   = ytd_end - pd.DateOffset(years=1)
        py_ytd_sum, _ = _sum_window(df_hs, py_ytd_start, py_ytd_end)

        g, flag = _growth(py_ytd_sum, ytd_sum, sanity_ratio)
        rows.append(dict(
            hs_code=hs, reference_date=ref_dt, metric_type="ytd_growth",
            metric_date=ref_dt,
            period_label=f"{ytd_start:%Y-%m}~{ytd_end:%Y-%m} vs {py_ytd_start:%Y-%m}~{py_ytd_end:%Y-%m}",
            base_value=py_ytd_sum, compare_value=ytd_sum,
            growth_rate=g, flag=flag,
        ))

        # ── 4. 올해 예상성장률: 올해 1~12월 전체(실측+예측) vs 작년 1~12월 전체 ──
        fy_start = pd.Timestamp(year=cur_year, month=1, day=1)
        fy_end   = pd.Timestamp(year=cur_year, month=12, day=31)
        fy_sum, fy_n = _sum_window(df_hs, fy_start, fy_end)
        fy_complete = fy_n >= 12

        py_start = fy_start - pd.DateOffset(years=1)
        py_end   = fy_end - pd.DateOffset(years=1)
        py_sum, _ = _sum_window(df_hs, py_start, py_end)

        g, flag = _growth(py_sum, fy_sum, sanity_ratio)
        if flag is None and not fy_complete:
            flag = "incomplete_year"
        rows.append(dict(
            hs_code=hs, reference_date=ref_dt, metric_type="full_year_growth",
            metric_date=fy_end,
            period_label=f"{fy_start:%Y}(실측+예측, {fy_n}개월) vs {py_start:%Y}(실측)",
            base_value=py_sum, compare_value=fy_sum,
            growth_rate=g, flag=flag,
        ))

        # ── 5. 작년 성장률: 작년 1~12월 vs 재작년 1~12월 (완전 확정 과거) ──
        ly_start = pd.Timestamp(year=cur_year - 1, month=1, day=1)
        ly_end   = pd.Timestamp(year=cur_year - 1, month=12, day=31)
        ly_sum, _ = _sum_window(df_hs, ly_start, ly_end)

        lly_start = ly_start - pd.DateOffset(years=1)
        lly_end   = ly_end - pd.DateOffset(years=1)
        lly_sum, _ = _sum_window(df_hs, lly_start, lly_end)

        g, flag = _growth(lly_sum, ly_sum, sanity_ratio)
        rows.append(dict(
            hs_code=hs, reference_date=ref_dt, metric_type="last_year_growth",
            metric_date=ly_end,
            period_label=f"{ly_start:%Y} vs {lly_start:%Y}",
            base_value=lly_sum, compare_value=ly_sum,
            growth_rate=g, flag=flag,
        ))

        # ── 6/7. MoM / YoY: 기준월 전후 window_months 구간의 매월 시계열 ──
        ts_start = _month_end(ref_dt - pd.DateOffset(months=window_months))
        ts_end   = _month_end(ref_dt + pd.DateOffset(months=window_months))
        ts = df_hs.set_index("date_month_end")["export_value"]

        month_range = pd.date_range(ts_start, ts_end, freq="MS") + pd.offsets.MonthEnd(0)
        for m in month_range:
            cur_val = ts.get(m, np.nan)

            prev_m = (m - pd.DateOffset(months=1))
            prev_m = prev_m + pd.offsets.MonthEnd(0)
            prev_val = ts.get(prev_m, np.nan)
            g_mom, flag_mom = _growth(prev_val, cur_val, sanity_ratio)
            rows.append(dict(
                hs_code=hs, reference_date=ref_dt, metric_type="mom_growth",
                metric_date=m,
                period_label=f"{m:%Y-%m} vs {prev_m:%Y-%m}",
                base_value=prev_val, compare_value=cur_val,
                growth_rate=g_mom, flag=flag_mom,
            ))

            prev_y = (m - pd.DateOffset(years=1))
            prev_y = prev_y + pd.offsets.MonthEnd(0)
            prev_y_val = ts.get(prev_y, np.nan)
            g_yoy, flag_yoy = _growth(prev_y_val, cur_val, sanity_ratio)
            rows.append(dict(
                hs_code=hs, reference_date=ref_dt, metric_type="yoy_growth",
                metric_date=m,
                period_label=f"{m:%Y-%m} vs {prev_y:%Y-%m}",
                base_value=prev_y_val, compare_value=cur_val,
                growth_rate=g_yoy, flag=flag_yoy,
            ))

    long_df = pd.DataFrame(rows)

    n_total = len(long_df)
    n_valid = int(long_df["growth_rate"].notna().sum())
    print("\n" + "=" * 80)
    print(f"총 {n_total:,} 행 생성 (HS {len(hs_codes):,}개 x 지표 {long_df['metric_type'].nunique()}종) "
          f"/ 유효 growth_rate {n_valid:,}건")
    print("\n[metric_type 별 건수 / 유효 건수]")
    summary = long_df.groupby("metric_type").agg(
        rows=("growth_rate", "size"),
        valid=("growth_rate", lambda s: s.notna().sum()),
    )
    print(summary.to_string())

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        ref_tag = pd.Timestamp.today().strftime("%Y%m%d")
        fp = os.path.join(save_path, f"export_growth_long_{ref_tag}.csv")
        long_df.to_csv(fp, index=False, encoding="utf-8-sig")
        print(f"\n[저장] {fp} ({len(long_df):,} rows)")

    return long_df


In [3]:

# 사용 예시
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

save_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_Export_growth"

long_df = build_export_growth_long(
    db_info=db_info,
    reference_date=None,      # None -> HS코드별 최신 실측월을 기준월로 자동 사용
    window_months=12,
    save_path=save_path,
)

long_df.head(20)


수출 성장률 종합 분석 (long-format)
데이터 로딩 중...
[버전 관리] 'created_at' 기준 HS별 최신 run 만 사용
  로드 레코드: 68,818 / HS 코드: 500

총 26,394 행 생성 (HS 500개 x 지표 7종) / 유효 growth_rate 26,394건

[metric_type 별 건수 / 유효 건수]
                    rows  valid
metric_type                    
full_year_growth     498    498
future_12m_growth    498    498
last_year_growth     498    498
mom_growth         11952  11952
past_12m_growth      498    498
yoy_growth         11952  11952
ytd_growth           498    498

[저장] C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_Export_growth\export_growth_long_20260724.csv (26,394 rows)


,hs_code,reference_date,metric_type,metric_date,period_label,base_value,compare_value,growth_rate,flag
0,020130,2026-05-31,past_12m_growth,2026-05-31,2025-06~2026-05 vs 2024-06~2025-05,4.056005e+09,3.652743e+09,-9.942342,None
1,020130,2026-05-31,future_12m_growth,2027-05-31,2026-06~2027-05 vs 2025-06~2026-05,3.652743e+09,3.692424e+09,1.086326,None
2,020130,2026-05-31,ytd_growth,2026-05-31,2026-01~2026-05 vs 2025-01~2025-05,1.600826e+09,1.475708e+09,-7.815828,None
3,020130,2026-05-31,full_year_growth,2026-12-31,"2026(실측+예측, 12개월) vs 2025(실측)",3.777861e+09,3.638809e+09,-3.680700,None
4,020130,2026-05-31,last_year_growth,2025-12-31,2025 vs 2024,4.141884e+09,3.777861e+09,-8.788822,None
5,020130,2026-05-31,mom_growth,2025-06-30,2025-06 vs 2025-05,3.031383e+08,3.115293e+08,2.768033,None
6,020130,2026-05-31,yoy_growth,2025-06-30,2025-06 vs 2024-06,3.918857e+08,3.115293e+08,-20.505076,None
7,020130,2026-05-31,mom_growth,2025-07-31,2025-07 vs 2025-06,3.115293e+08,3.225977e+08,3.552925,None
8,020130,2026-05-31,yoy_growth,2025-07-31,2025-07 vs 2024-07,3.623714e+08,3.225977e+08,-10.975962,None
9,020130,2026-05-31,mom_growth,2025-08-31,2025-08 vs 2025-07,3.225977e+08,3.094247e+08,-4.083388,None


In [6]:
def rank_future_12m_growth(long_df, save_path=None, top_n=20):
    """
    long_df(build_export_growth_long 결과)에서 metric_type == 'future_12m_growth' 만 뽑아
    전체 HS 코드의 향후 12개월 수출 성장률 랭킹을 만든다.

    정상(flag 없음) 항목을 growth_rate 내림차순으로 먼저 배치하고,
    이상치(flag 있음: base<=0, explosive 등)는 하단에 배치한다.
    """
    fut = long_df[long_df["metric_type"] == "future_12m_growth"].copy()

    fut = fut.rename(columns={
        "base_value": "recent_12month",     # 기준월 포함 최근 12개월 합
        "compare_value": "future_12month",  # 기준월 이후 12개월(예측 포함) 합
    })

    fut["_is_clean"] = fut["flag"].isna()
    fut = (fut.sort_values(["_is_clean", "growth_rate"], ascending=[False, False], na_position="last")
              .drop(columns="_is_clean")
              .reset_index(drop=True))
    fut.insert(0, "rank", range(1, len(fut) + 1))

    out_cols = ["rank", "hs_code", "reference_date", "period_label",
                "recent_12month", "future_12month", "growth_rate", "flag"]
    ranking_df = fut[out_cols]

    n_clean = int(ranking_df["flag"].isna().sum())
    print("=" * 80)
    print(f"향후 12개월 수출 성장률 랭킹 "
          f"(전체 {len(ranking_df):,}개 HS 코드 / 정상 {n_clean:,} / 이상 {len(ranking_df) - n_clean:,})")
    print("=" * 80)

    clean = ranking_df[ranking_df["flag"].isna()]
    with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
        print(f"\n[상위 {top_n} — 정상 항목만, 단위: %]")
        print(clean.head(top_n)[
            ["rank", "hs_code", "recent_12month", "future_12month", "growth_rate"]
        ].to_string(index=False))

    if n_clean:
        print(f"\n[정상 성장률 통계] 평균 {clean['growth_rate'].mean():.2f}% / "
              f"중앙값 {clean['growth_rate'].median():.2f}% / "
              f"최대 {clean['growth_rate'].max():.2f}% / "
              f"최소 {clean['growth_rate'].min():.2f}%")

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        fp = os.path.join(save_path, f"future_12m_growth_ranking_{pd.Timestamp.today():%Y%m%d}.csv")
        ranking_df.to_csv(fp, index=False, encoding="utf-8-sig")
        print(f"\n[저장] {fp} ({len(ranking_df):,} rows)")

    return ranking_df


# 사용 예시 (long_df, save_path 는 이전 셀에서 이미 만들어져 있다고 가정)
future_ranking_df = rank_future_12m_growth(long_df, save_path=save_path)
future_ranking_df.head(20)

향후 12개월 수출 성장률 랭킹 (전체 498개 HS 코드 / 정상 498 / 이상 0)

[상위 20 — 정상 항목만, 단위: %]
 rank hs_code     recent_12month     future_12month  growth_rate
    1  270900 119,653,349,782.00 262,008,565,518.43       118.97
    2  120190  18,659,251,196.00  34,975,190,637.69        87.44
    3  847150  43,488,125,031.00  76,402,364,993.03        75.69
    4  271019  75,287,256,575.00 124,976,669,635.97        66.00
    5  852351  12,015,931,913.00  19,446,118,837.66        61.84
    6  841989   1,003,694,844.00   1,614,802,843.84        60.89
    7  854449   3,740,770,664.00   5,897,467,711.61        57.65
    8  271113   5,372,500,200.00   8,374,081,934.85        55.87
    9  841112   3,490,217,823.00   5,263,812,397.01        50.82
   10  870451   1,052,372,766.00   1,570,695,095.88        49.25
   11  853690   4,265,965,200.00   6,354,517,266.23        48.96
   12  290110   7,676,793,509.00  11,362,776,476.41        48.01
   13  711011   3,058,624,212.00   4,487,311,753.15        46.71
   14  711031  

,rank,hs_code,reference_date,period_label,recent_12month,future_12month,growth_rate,flag
0,1,270900,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,1.196533e+11,2.620086e+11,118.973030,None
1,2,120190,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,1.865925e+10,3.497519e+10,87.441555,None
2,3,847150,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,4.348813e+10,7.640236e+10,75.685581,None
3,4,271019,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,7.528726e+10,1.249767e+11,65.999766,None
4,5,852351,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,1.201593e+10,1.944612e+10,61.836127,None
5,6,841989,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,1.003695e+09,1.614803e+09,60.885836,None
6,7,854449,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,3.740771e+09,5.897468e+09,57.653816,None
7,8,271113,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,5.372500e+09,8.374082e+09,55.869365,None
8,9,841112,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,3.490218e+09,5.263812e+09,50.816157,None
9,10,870451,2026-05-31,2026-06~2027-05 vs 2025-06~2026-05,1.052373e+09,1.570695e+09,49.252731,None


In [8]:
def get_hs_timeseries_growth(db_info, hs_code,
                              table="us_trade_export_monthly_with_forecast"):
    """
    특정 HS 코드의 전체 시계열(실측+예측)을 불러와
    MoM growth / YoY growth 를 나란히 붙인 DataFrame 반환.

    Returns
    -------
    ts_df : pd.DataFrame
        columns = [date_month_end, is_forecast, expDlr, expDlr_forecast,
                   export_value, mom_growth, mom_flag, yoy_growth, yoy_flag]
    """
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info.get('port', 3306)}/{db_info['database']}"
    )

    with engine.connect() as conn:
        vcol = _detect_version_col(conn, table)
        if vcol:
            sql = f"""
                SELECT t.hs_code, t.date_month_end, t.expDlr,
                       t.expDlr_forecast, t.is_forecast
                FROM {table} t
                JOIN (
                    SELECT hs_code, MAX({vcol}) AS mx
                    FROM {table}
                    WHERE hs_code = :hs_code
                    GROUP BY hs_code
                ) l ON t.hs_code = l.hs_code AND t.{vcol} = l.mx
                WHERE t.hs_code = :hs_code
                ORDER BY t.date_month_end
            """
        else:
            sql = f"""
                SELECT hs_code, date_month_end, expDlr, expDlr_forecast, is_forecast
                FROM {table}
                WHERE hs_code = :hs_code
                ORDER BY date_month_end
            """
        res  = conn.execute(text(sql), {"hs_code": hs_code})
        rows = res.fetchall()
        cols = list(res.keys())
    engine.dispose()

    df = pd.DataFrame(rows, columns=cols)
    if df.empty:
        print(f"❌ HS Code {hs_code} 데이터가 없습니다.")
        return df

    df["date_month_end"] = pd.to_datetime(df["date_month_end"])
    df = df.sort_values("date_month_end").reset_index(drop=True)
    df["export_value"] = np.where(df["is_forecast"] == 1,
                                   df["expDlr_forecast"], df["expDlr"])

    ts = df.set_index("date_month_end")["export_value"]

    mom_growth, mom_flag = [], []
    yoy_growth, yoy_flag = [], []
    for d in df["date_month_end"]:
        prev_m = _month_end(d - pd.DateOffset(months=1))
        g, f = _growth(ts.get(prev_m, np.nan), ts.get(d, np.nan))
        mom_growth.append(g); mom_flag.append(f)

        prev_y = _month_end(d - pd.DateOffset(years=1))
        g, f = _growth(ts.get(prev_y, np.nan), ts.get(d, np.nan))
        yoy_growth.append(g); yoy_flag.append(f)

    df["mom_growth"] = mom_growth
    df["mom_flag"]   = mom_flag
    df["yoy_growth"] = yoy_growth
    df["yoy_flag"]   = yoy_flag

    ts_df = df[["date_month_end", "is_forecast", "expDlr", "expDlr_forecast",
                "export_value", "mom_growth", "mom_flag", "yoy_growth", "yoy_flag"]]

    n_actual = int((ts_df["is_forecast"] == 0).sum())
    n_forecast = int((ts_df["is_forecast"] == 1).sum())
    print(f"HS {hs_code} : 실측 {n_actual}개월 / 예측 {n_forecast}개월 (총 {len(ts_df)}개월)")

    return ts_df


# 사용 예시
hs_check = '340242'
ts_df = get_hs_timeseries_growth(db_info, hs_check)
ts_df

HS 340242 : 실측 53개월 / 예측 18개월 (총 71개월)


,date_month_end,is_forecast,expDlr,expDlr_forecast,export_value,mom_growth,mom_flag,yoy_growth,yoy_flag
0,2022-01-31,0,63860791.0,6.386079e+07,6.386079e+07,NaN,missing,NaN,missing
1,2022-02-28,0,71901565.0,7.190156e+07,7.190156e+07,12.591097,None,NaN,missing
2,2022-03-31,0,66744560.0,6.674456e+07,6.674456e+07,-7.172313,None,NaN,missing
3,2022-04-30,0,76061483.0,7.606148e+07,7.606148e+07,13.959075,None,NaN,missing
4,2022-05-31,0,74736323.0,7.473632e+07,7.473632e+07,-1.742222,None,NaN,missing
...,...,...,...,...,...,...,...,...,...
66,2027-07-31,1,NaN,7.445861e+07,7.445861e+07,-0.522657,None,0.487745,None
67,2027-08-31,1,NaN,6.692998e+07,6.692998e+07,-10.111157,None,0.487745,None
68,2027-09-30,1,NaN,6.899541e+07,6.899541e+07,3.085952,None,0.487745,None
69,2027-10-31,1,NaN,6.221230e+07,6.221230e+07,-9.831247,None,0.487745,None


In [10]:
def get_hs_timeseries_growth(db_info, hs_code,
                              table="us_trade_export_monthly_with_forecast"):
    """
    특정 HS 코드의 전체 시계열(실측+예측)을 불러와
    MoM growth / YoY growth 를 나란히 붙인 DataFrame 반환.

    Returns
    -------
    ts_df : pd.DataFrame
        columns = [date_month_end, is_forecast, expDlr, expDlr_forecast,
                   export_value, mom_growth, mom_flag, yoy_growth, yoy_flag]
    """
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info.get('port', 3306)}/{db_info['database']}"
    )

    with engine.connect() as conn:
        vcol = _detect_version_col(conn, table)
        if vcol:
            sql = f"""
                SELECT t.hs_code, t.date_month_end, t.expDlr,
                       t.expDlr_forecast, t.is_forecast
                FROM {table} t
                JOIN (
                    SELECT hs_code, MAX({vcol}) AS mx
                    FROM {table}
                    WHERE hs_code = :hs_code
                    GROUP BY hs_code
                ) l ON t.hs_code = l.hs_code AND t.{vcol} = l.mx
                WHERE t.hs_code = :hs_code
                ORDER BY t.date_month_end
            """
        else:
            sql = f"""
                SELECT hs_code, date_month_end, expDlr, expDlr_forecast, is_forecast
                FROM {table}
                WHERE hs_code = :hs_code
                ORDER BY date_month_end
            """
        res  = conn.execute(text(sql), {"hs_code": hs_code})
        rows = res.fetchall()
        cols = list(res.keys())
    engine.dispose()

    df = pd.DataFrame(rows, columns=cols)
    if df.empty:
        print(f"❌ HS Code {hs_code} 데이터가 없습니다.")
        return df

    df["date_month_end"] = pd.to_datetime(df["date_month_end"])
    df = df.sort_values("date_month_end").reset_index(drop=True)
    df["export_value"] = np.where(df["is_forecast"] == 1,
                                   df["expDlr_forecast"], df["expDlr"])

    ts = df.set_index("date_month_end")["export_value"]

    mom_growth, mom_flag = [], []
    yoy_growth, yoy_flag = [], []
    for d in df["date_month_end"]:
        prev_m = _month_end(d - pd.DateOffset(months=1))
        g, f = _growth(ts.get(prev_m, np.nan), ts.get(d, np.nan))
        mom_growth.append(g); mom_flag.append(f)

        prev_y = _month_end(d - pd.DateOffset(years=1))
        g, f = _growth(ts.get(prev_y, np.nan), ts.get(d, np.nan))
        yoy_growth.append(g); yoy_flag.append(f)

    df["mom_growth"] = mom_growth
    df["mom_flag"]   = mom_flag
    df["yoy_growth"] = yoy_growth
    df["yoy_flag"]   = yoy_flag

    ts_df = df[["date_month_end", "is_forecast", "expDlr", "expDlr_forecast",
                "export_value", "mom_growth", "mom_flag", "yoy_growth", "yoy_flag"]]

    n_actual = int((ts_df["is_forecast"] == 0).sum())
    n_forecast = int((ts_df["is_forecast"] == 1).sum())
    print(f"HS {hs_code} : 실측 {n_actual}개월 / 예측 {n_forecast}개월 (총 {len(ts_df)}개월)")

    return ts_df


# 사용 예시
hs_check = '854231'
ts_df = get_hs_timeseries_growth(db_info, hs_check)
ts_df

HS 854231 : 실측 125개월 / 예측 18개월 (총 143개월)


,date_month_end,is_forecast,expDlr,expDlr_forecast,export_value,mom_growth,mom_flag,yoy_growth,yoy_flag
0,2016-01-31,0,1.418644e+09,1.418644e+09,1.418644e+09,NaN,missing,NaN,missing
1,2016-02-29,0,1.473350e+09,1.473350e+09,1.473350e+09,3.856261,None,NaN,missing
2,2016-03-31,0,1.942899e+09,1.942899e+09,1.942899e+09,31.869448,None,NaN,missing
3,2016-04-30,0,1.562209e+09,1.562209e+09,1.562209e+09,-19.593891,None,NaN,missing
4,2016-05-31,0,1.743510e+09,1.743510e+09,1.743510e+09,11.605427,None,NaN,missing
...,...,...,...,...,...,...,...,...,...
138,2027-07-31,1,NaN,3.475358e+09,3.475358e+09,6.739306,None,6.989788,None
139,2027-08-31,1,NaN,3.329496e+09,3.329496e+09,-4.197027,None,6.989788,None
140,2027-09-30,1,NaN,3.228048e+09,3.228048e+09,-3.046939,None,6.989788,None
141,2027-10-31,1,NaN,3.436089e+09,3.436089e+09,6.444774,None,6.989788,None


In [7]:

# 특정 HS 코드 지표 확인 (wide 형태로 펼쳐보기)
hs_check = '852351'
pivot_check = (
    long_df[long_df['hs_code'] == hs_check]
    .pivot_table(index='metric_type', values='growth_rate', aggfunc='first')
)
print(pivot_check)

# MoM / YoY 시계열만 별도로 보기
ts_view = (
    long_df[(long_df['hs_code'] == hs_check) & (long_df['metric_type'].isin(['mom_growth', 'yoy_growth']))]
    .sort_values(['metric_type', 'metric_date'])
    [['metric_type', 'metric_date', 'base_value', 'compare_value', 'growth_rate', 'flag']]
)
ts_view


                   growth_rate
metric_type                   
full_year_growth     88.266131
future_12m_growth    61.836127
last_year_growth     28.058481
mom_growth           21.362103
past_12m_growth      46.441167
yoy_growth           39.078684
ytd_growth           80.293653


,metric_type,metric_date,base_value,compare_value,growth_rate,flag
18661,mom_growth,2025-06-30,6.601753e+08,8.012026e+08,21.362103,None
18663,mom_growth,2025-07-31,8.012026e+08,6.442592e+08,-19.588479,None
18665,mom_growth,2025-08-31,6.442592e+08,7.865921e+08,22.092501,None
18667,mom_growth,2025-09-30,7.865921e+08,6.706961e+08,-14.733947,None
18669,mom_growth,2025-10-31,6.706961e+08,6.667398e+08,-0.589881,None
18671,mom_growth,2025-11-30,6.667398e+08,7.659933e+08,14.886402,None
18673,mom_growth,2025-12-31,7.659933e+08,1.065144e+09,39.053904,None
18675,mom_growth,2026-01-31,1.065144e+09,1.093694e+09,2.680386,None
18677,mom_growth,2026-02-28,1.093694e+09,1.457199e+09,33.236471,None
18679,mom_growth,2026-03-31,1.457199e+09,1.361982e+09,-6.534229,None
